# MammoRisk+ Module 2: PRS-313 on Colab (O1)

Runs the whole Module 2 pipeline on Colab, so nothing has to be installed on the laptop:

1. **samples**: gnomAD HGDP+1KG metadata (3.4 MB) → CEU + PJL + 8 HGDP Pakistani groups, high quality, unrelated
2. **regions / extract**: `bcftools view -R` pulls **only the 313 PRS positions** from the public gnomAD VCFs
   (3.63 TB in total; we download a few MB) → plink2 fileset
3. **harmonise / score / dosage**: allele matching, `plink2 --score`, effect-allele dosages
4. **analyse**: % reclassified European (CEU) vs Pakistani reference with bootstrap CIs, drivers, sensitivity analyses

Data: gnomAD v3.1.2 HGDP+1KG subset, GRCh38 (DECISIONS D-19). Weights: PGS000004 GRCh38 harmonised.
Runtime: CPU is enough. Expect ~10–30 min, mostly the remote extraction.

## 1. Get the code
**Option A** (needs today's commits pushed to GitHub): run the next cell.
**Option B** (nothing pushed): on the laptop run
`git archive -o mammorisk.zip HEAD` in the repo, then upload `mammorisk.zip` with the second cell.

In [ ]:
# Option A: clone the public repo
!git clone --depth 1 https://github.com/FaiqaZarar/MammoRisk-Plus.git /content/MammoRisk-Plus

In [ ]:
# Option B: upload mammorisk.zip made with `git archive -o mammorisk.zip HEAD`
# from google.colab import files
# up = files.upload()
# !mkdir -p /content/MammoRisk-Plus && unzip -o -q mammorisk.zip -d /content/MammoRisk-Plus

In [ ]:
%cd /content/MammoRisk-Plus
!git log --oneline -1 2>/dev/null || echo "(zip upload, no git history)"

## 2. Install tools (bcftools from Ubuntu, plink2 pinned build)

In [ ]:
!apt-get -qq update && apt-get -qq install -y bcftools > /dev/null
# PLINK 2.0 Beta 1.3 (31 Oct 2026), generic x86_64 Linux build from cog-genomics.org/plink/2.0
!wget -q https://s3.amazonaws.com/plink2-assets/beta1/plink2_linux_x86_64_20261031.zip -O /tmp/plink2.zip   && unzip -o -q /tmp/plink2.zip -d /usr/local/bin && chmod +x /usr/local/bin/plink2
!bcftools --version | head -2; plink2 --version
!pip -q install pyyaml pandas scipy scikit-learn

## 3. Run the pipeline
`--data-root /content/data` keeps genotypes in the Colab VM (deleted when the session ends).
Every step can be re-run on its own (`--step <name>`). The extraction skips chromosomes already downloaded.

In [ ]:
CFG = "configs/m2_prs.yaml"
ARGS = f"--config {CFG} --data-root /content/data --results-dir /content/results/m2_prs"
!python -m module2_prs.run {ARGS} --step samples
!python -m module2_prs.run {ARGS} --step regions

In [ ]:
# Remote extraction: 22 chromosomes, only the 313 positions, 394 samples
!python -m module2_prs.run {ARGS} --step extract

In [ ]:
!python -m module2_prs.run {ARGS} --step harmonise
!python -m module2_prs.run {ARGS} --step score
!python -m module2_prs.run {ARGS} --step dosage

In [ ]:
# Main analysis + bootstrap (1,000 resamples; a few minutes)
!python -m module2_prs.run {ARGS} --step analyse

## 4. Look at the results

In [ ]:
import json, pandas as pd
R = "/content/results/m2_prs"
print(json.dumps(json.load(open(f"{R}/run_info_harmonise.json"))["coverage"], indent=1))
display(pd.read_csv(f"{R}/sample_counts.csv"))
display(pd.read_csv(f"{R}/reclassification.csv"))
display(pd.read_csv(f"{R}/crosstab_european_vs_pakistani.csv", index_col=0))
display(pd.read_csv(f"{R}/own_group_sensitivity.csv"))
display(pd.read_csv(f"{R}/driver_snps.csv").head(15))

## 5. Download the results
The zip holds aggregate tables plus per-person scores and dosages of **public** 1000G/HGDP samples.
Keep it in `results/` locally (gitignored). Record the headline numbers in `docs/STATUS.md` with this
config and the date.

In [ ]:
!cd /content/results && zip -q -r m2_prs_results.zip m2_prs
from google.colab import files
files.download("/content/results/m2_prs_results.zip")